# NLP â€” evaluation and error analysis

Interactive inspection of the already executed baseline and learned-presence experiments. Existing metrics, predictions, split definitions, and audit statuses are read-only inputs; no evaluation report or prediction file is created here.

In [1]:
import sys
sys.dont_write_bytecode = True  # no __pycache__/ or .pyc; must run before project imports

from pathlib import Path
import csv
import json
import sys
from collections import Counter
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
ROOT = next(
    (p for p in (current, *current.parents) if (p / "src" / "liversira_ai").is_dir()),
    None
)

if ROOT is None:
    COLAB_ROOT = Path("/content/drive/MyDrive/LiverSira AI/Training")
    if (COLAB_ROOT / "src" / "liversira_ai").is_dir():
        ROOT = COLAB_ROOT

SRC_DIR = ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RESULTS = ROOT / 'results/nlp'

def read_json(path): 
    return json.loads(path.read_text(encoding='utf-8'))

def read_jsonl(path): 
    return [json.loads(line) for line in path.open(encoding='utf-8') if line.strip()]

SPLITS_PATH = ROOT / 'data/processed/nlp/splits.json'

splits = read_json(SPLITS_PATH)
print({"project_root": str(ROOT)})
print({name: len(ids) for name, ids in splits.items()})

{'project_root': 'D:\\University\\Graduation Project\\LiverSira AI'}
{'train': 6482, 'val': 1389, 'test': 1390}


## Existing baseline evaluation

In [2]:
baseline_metrics = read_json(RESULTS / 'baseline/baseline_metrics_updated.json')
display(pd.json_normalize(baseline_metrics, sep='.').T.rename(columns={0: 'value'}).head(40))
baseline_predictions = read_jsonl(RESULTS / 'predictions/baseline_inferences_updated.jsonl')
prediction_summary = pd.DataFrame([{'records': len(baseline_predictions), 'has_lesion_true': sum(item['liver_lesions']['has_lesion'] is True for item in baseline_predictions), 'has_lesion_false': sum(item['liver_lesions']['has_lesion'] is False for item in baseline_predictions), 'abstained': sum(item['liver_lesions']['has_lesion'] is None for item in baseline_predictions)}])
display(prediction_summary)

,value
comparison_basis,source_labels_not_clinical_gold_standard
inference_file,results\nlp\predictions\baseline_inferences_up...
splits_file,data\processed\nlp\splits.json
id_validation.ground_truth_count,9262
id_validation.inference_count,9262
id_validation.ground_truth_duplicates,[]
id_validation.inference_duplicates,[]
id_validation.split_ids_missing_ground_truth,[]
id_validation.split_ids_missing_inference,[]
id_validation.ground_truth_ids_missing_from_splits,[BDMAP_00005266]


,records,has_lesion_true,has_lesion_false,abstained
0,9262,1506,2145,5611


## Frozen model metrics

In [3]:
learned_metrics = read_json(RESULTS / 'final_test/learned_presence_test_metrics.json')
validation_metrics = read_json(RESULTS / 'learned_presence_validation/learned_presence_validation_metrics.json')
comparison = pd.DataFrame([
    {'experiment': 'learned_presence_validation', **{key: validation_metrics.get(key) for key in ['test_studies', 'evaluated', 'accuracy', 'precision', 'recall', 'f1', 'macro_f1']}},
    {'experiment': 'learned_presence_final_test', **{key: learned_metrics.get(key) for key in ['test_studies', 'evaluated', 'accuracy', 'precision', 'recall', 'f1', 'macro_f1']}},
])
display(comparison)

,experiment,test_studies,evaluated,accuracy,precision,recall,f1,macro_f1
0,learned_presence_validation,NaN,None,None,None,None,None,None
1,learned_presence_final_test,1390.0,None,None,None,None,None,None


## Split and leakage context

In [4]:
audit_path = ROOT / 'data/processed/nlp/target_consistency_audit.csv'
with audit_path.open(encoding='utf-8-sig', newline='') as handle:
    audit = pd.DataFrame(list(csv.DictReader(handle)))
display(audit['overall_status'].value_counts(dropna=False).rename('studies').to_frame())
split_rows = pd.DataFrame([(split, record_id) for split, ids in splits.items() for record_id in ids], columns=['split', 'id'])
display(split_rows.groupby('split').size().rename('studies').to_frame())
print({'cross_split_id_overlap': len(set(splits['train']) & set(splits['val']) | set(splits['train']) & set(splits['test']) | set(splits['val']) & set(splits['test'])), 'patient_level_leakage': 'not assessable: no patient identifier in NLP master records'})

,studies
overall_status,
UNCERTAIN,6067
MATCH,2944
MISMATCH,251


,studies
split,
test,1390
train,6482
val,1389


{'cross_split_id_overlap': 0, 'patient_level_leakage': 'not assessable: no patient identifier in NLP master records'}


## Inline error and uncertainty examples

In [5]:
gt = {item['id']: item for item in read_jsonl(ROOT / 'data/processed/nlp/ground_truth.jsonl')}
errors = []
for item in baseline_predictions:
    truth = gt.get(item['id'], {}).get('ground_truth', {})
    prediction = item['liver_lesions'].get('has_lesion')
    if prediction is None or (truth and prediction != truth.get('has_lesion')):
        errors.append({'id': item['id'], 'source_label': truth.get('has_lesion'), 'prediction': prediction, 'category': 'abstention' if prediction is None else 'presence_error'})
display(pd.DataFrame(errors).head(20))
print({'abstentions_or_presence_errors': len(errors), 'baseline_records_inspected': len(baseline_predictions)})

,id,source_label,prediction,category
0,BDMAP_00000002,False,None,abstention
1,BDMAP_00000003,False,None,abstention
2,BDMAP_00000004,False,True,presence_error
3,BDMAP_00000008,False,None,abstention
4,BDMAP_00000010,False,None,abstention
5,BDMAP_00000011,False,None,abstention
6,BDMAP_00000015,False,None,abstention
7,BDMAP_00000016,False,None,abstention
8,BDMAP_00000017,False,None,abstention
9,BDMAP_00000018,False,None,abstention


{'abstentions_or_presence_errors': 5649, 'baseline_records_inspected': 9262}


## Summary

The notebook separates source-derived evaluation from clinical validation, preserves the frozen train/validation/test protocol, and makes abstentions and audit groups visible. It does not retrain models, use the test split for configuration, or write new result files.

In [6]:
assert set(splits) == {'train', 'val', 'test'}
assert not (set(splits['train']) & set(splits['val']))
assert not (set(splits['train']) & set(splits['test']))
assert not (set(splits['val']) & set(splits['test']))
print('Evaluation inspection completed without writing files.')

Evaluation inspection completed without writing files.
